## uwtools + ecFlow

The notebook demonstrates the use of `uwtools` to configure and run an ecFlow server, to generate an ecFlow workflow (a suite definition an `.ecf` files), and to load the suite into the server and execute it. As a demo application, we perform verification of 2-meter temperature at the 6-hour leadtime of a GFS forecast against a GFS leadtime-zero forecast as a source of truth.

First, get to the correct working directory and activate the virtual environment for the demo:

In [ ]:
cd $BASEDIR
source conda/etc/profile.d/conda.sh
conda activate demo

Here's the `uwtools`-enabled configuration file for the demo. It's not necessary to understand all of it now. The top-level `app` block defines various application-level items that are useful throughout the workflow. The `ecflow` block defines server and suite-definition blocks specific to ecFlow. And the `wxvx` block defines parameters for verification to be performed by [wxvx](https://github.com/NOAA-GSL/wxvx).

One thing to note looking at the full config is the use of [Jinja2](https://jinja.palletsprojects.com/en/stable/templates/) expressions to correctly tie together various config blocks.

In [ ]:
cat config.yaml

The `ecflow.server` block provides the bare mininum for running the ecFlow server: A definition of the `ECF_HOME` environment variable, which specifies where the server should run and where, by default, it will look for `.h` include files when creating job scripts from `.ecf` files:

In [ ]:
uw config realize -i config.yaml --key-path ecflow.server

In this case, `ecflow.server` refers to `app.basedir` for its value, and `app.basedir` refers to the environment variable `BASEDIR` set by the `start` script.

Start the ecFlow server using the config specified in `config.yaml`. Request a JSON-formatted report of important server environment variables, redirect that to `server.json`, and redirect the rest of the output to `server.log`:

In [ ]:
uw ecflow server -c config.yaml --report >server.json 2>server.log &

Wait until the server has started up and written `server.json`:

In [ ]:
while [[ ! -s server.json ]]; do sleep 1; done

View `server.log` and note that there were no errors, and the the server is using a random available TCP port.

Also note that, since `~/.ecflowrc/ssl` did not already contain the SSL certificate files required for secure communication with the ecFlow server, these were generated.

In [ ]:
cat server.log

At server-startup time, `uw ecflow server` also creates a `.h` include file called `server.h`, defining environment variables that will be needed by `.ecf` files:

In [ ]:
cat server.h

The `server.json` file, containing current values for some useful server-related environment variables, was also created:

In [ ]:
jq . server.json

Use `jq` to set environment variables based on these values. extract values from this file to use to set some crucial environment variables that will allow the ecFlow client tool, `ecflow_client`, to successfully communicate with the server:

In [ ]:
for key in $(jq -r keys[] server.json); do
  export $key="$(jq -r .$key server.json)"
  echo $key: ${!key}
done

With these environment variables in place, use `ecflow_client` to ping the server and verify that it is listening:

In [ ]:
ecflow_client --ping

By default, the server starts halted, so place it into running state with the `restart` command:

In [ ]:
ecflow_client --stats | grep Status
ecflow_client --restart
ecflow_client --stats | grep Status

Now the server is ready to be loaded with a suite definition, provided by the `ecflow.suitedef` block in `config.yaml`. The `uw ecflow realize` command creates a `suite.def` file in the directory specified by `--output-dir` and, under the directory specified by `--scripts-dir`, any `.ecf` files defined by `script` blocks in the suite definition:

In [ ]:
uw ecflow realize -c config.yaml --output-dir=$BASEDIR --scripts-dir=$BASEDIR

Next let's see the generated `suite.def` file, in ecFlow's custom configuration language. Note that the `fetch_truth` task can proceed immediately, but that `extract_grids` requires `fetch_truth` to be complete, as it uses the fetched truth data as an input; and that `verification` requires `extract_grids` to be complete, as it uses the extracted grids in verification:

In [ ]:
cat suite.def

The `.ecf` files, which are templates used to create executable job scripts that run ecFlow tasks, are created in a directory hierarchy based on the suite name, family names (this demo does not use families), and task names:

In [ ]:
tree vx

The `fetch_truth.ecf` file defines a task that will download truth data to verify a forecast against. It defines some help text between the `%manual` and `%end` tags that can be viewed, for example, in the ecFlow GUI application, `ecflow_ui`. When rendered by the server to a job script, the contents of the `server.h` and `common.h` include files will be inlined where the `%include` statements appear.

In [ ]:
cat vx/fetch_truth.ecf

We saw the contents of the `server.h` file generated by `uw ecflow server` earlier. Here's `common.h`, which belongs to this demo application and defines some functions and traps that allow task jobs to communicate their status to the server:

In [ ]:
cat common.h

Here's `extract_grids.ecf`, the `.ecf` file that runs `wxvx` to extract grids from the forecast and truth datasets to be used in verification:

In [ ]:
cat vx/extract_grids.ecf

Finally, we have `verification.ecf`, representing the final task in the workflow, which defines the task that performs verification with [MET](https://dtcenter.org/software-tools/model-evaluation-tools-met).

In [ ]:
cat vx/verification.ecf

Load the suite into the server:

In [ ]:
ecflow_client --load=suite.def

Verify that the suite was loaded:

In [ ]:
ecflow_client --get

Begin suite execution:

In [ ]:
ecflow_client --begin=vx

Wait for the tasks to complete:

In [ ]:
while true; do
  state=$(ecflow_client --query state /vx/fetch_truth)
  echo "  fetch_truth: $state"
  state=$(ecflow_client --query state /vx/extract_grids)
  echo "extract_grids: $state"
  test $state == complete && break
  sleep 1
done

Now that the workflow tasks are complete, examine some of the files created both by the server and by the workflow tasks.

First, note that the server has created, in the `ECF_HOME` directory, a log file based on the hostname of the system it is running on. These can be useful for debugging, but its contents are beyond the scope of this tutorial. Depending on when you look, you may also see a `.check` file, containg a checkpoint of the workflow state, which can be used to resume a workflow if the server is restarted.

In [ ]:
ls -l $ECF_HOST.$ECF_PORT.*

Let's look at the updated contents of the `vx/` directory:

In [ ]:
tree vx

We previously saw `extract_grids.ecf` and `fetch_truth.ecf` template files. The server rendered these to ready-to-run scripts `extract_grids.job1` and `fetch_truth.job1`, respectively. The `1` in the `.job1` filename extensions indicates that these files belong to the first of potentially multiple attempts to run this task.

The file `extract_grids.job1`, for example, is the concatenation of the header file `common.h`, the header file `server.h`, and the value of `ecflow.suitedef.suite_vs.task_extract_grids.script.body` from `config.yaml`. Here it is:

In [ ]:
cat vx/extract_grids.job1

The file `extract_grids.1` contains a log of the output of this job. Since our `common.h` performs a `set -x`, we see a trace of the execution of shell commands, as well as the log output of the `wxvx` utility as it runs.

In [ ]:
cat vx/extract_grids.1

The `forecast/` and `truth/` subdirectories contain the grids staged by `wxvx` for verification.

Shut down the server by simulating a CTRL-C that a command-line user would press:

In [ ]:
kill -INT %1
wait